# 01 - Exploratory Data Analysis & Implicit Feedback Distributions
This notebook performs exploratory data profiling on the Retailrocket e-commerce dataset:
- Event action distribution (view, addtocart, transaction)
- Matrix sparsity and user-item interaction density
- Activity tail profiling and bot/crawler detection
- Temporal timeline distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Load configuration
config_path = Path("../config.yaml")
if config_path.exists():
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    events_path = config.get("data", {}).get("raw_path", "../data/raw/events.csv")
else:
    events_path = "../data/raw/events.csv"

print(f"Reading events from: {events_path}")


In [ ]:
# Load events dataset
# Note: Retailrocket events.csv columns: timestamp, visitorid, event, itemid, transactionid
df_events = pd.read_csv(events_path)
print(f"Total interaction events: {len(df_events):,}")
df_events.head()


### 1. Event Type Distribution & Funnel Ratios

In [ ]:
event_counts = df_events['event'].value_counts()
event_pct = df_events['event'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({"Count": event_counts, "Percentage (%)": event_pct.round(2)})
print(summary_df)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=summary_df.index, y=summary_df["Count"], palette="Blues_d", ax=ax)
ax.set_title("Retailrocket Interaction Event Breakdown")
ax.set_ylabel("Total Occurrences")
plt.tight_layout()
plt.show()


### 2. Interaction Sparsity & Cardinality

In [ ]:
n_users = df_events['visitorid'].nunique()
n_items = df_events['itemid'].nunique()
n_events = len(df_events)

matrix_size = n_users * n_items
sparsity = (1.0 - (n_events / matrix_size)) * 100

print(f"Unique Visitors (Users): {n_users:,}")
print(f"Unique Items:            {n_items:,}")
print(f"Total Interactions:      {n_events:,}")
print(f"Matrix Sparsity:         {sparsity:.6f}%")


### 3. User Activity Profiling & Crawler Detection

In [ ]:
user_activity = df_events.groupby('visitorid').size()

print("User activity percentiles:")
print(user_activity.describe(percentiles=[0.5, 0.75, 0.90, 0.99, 0.999]))

# Detect extreme activity (crawlers/bots)
crawler_threshold = user_activity.quantile(0.999)
crawlers = user_activity[user_activity > crawler_threshold]
print(f"\nPotential scrapers/crawlers (> 99.9th percentile, > {crawler_threshold:.0f} events): {len(crawlers):,} users")

plt.figure(figsize=(8, 4))
sns.histplot(user_activity, bins=50, log_scale=(True, True), color="navy")
plt.title("Distribution of Interactions per Visitor (Log-Log Scale)")
plt.xlabel("Number of Events per Visitor")
plt.ylabel("Number of Visitors")
plt.show()


### 4. Temporal Distribution

In [ ]:
# Convert epoch milliseconds to datetime
df_events['datetime'] = pd.to_datetime(df_events['timestamp'], unit='ms')
df_events['date'] = df_events['datetime'].dt.date

daily_counts = df_events.groupby('date').size()

plt.figure(figsize=(12, 4))
daily_counts.plot(color='teal', lw=1.8)
plt.title("Daily Event Volume Over Time")
plt.xlabel("Date")
plt.ylabel("Events per Day")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Timeline starts: {df_events['datetime'].min()}")
print(f"Timeline ends:   {df_events['datetime'].max()}")
